# Data exploration

In [ ]:
import pandas as pd
import duckdb
import altair as alt

In [ ]:
conn = duckdb.connect(":memory:")
conn.execute("""CREATE TABLE customers AS SELECT * FROM '../data/customers.csv'""")
conn.execute("""CREATE TABLE subscriptions AS SELECT * FROM '../data/subscriptions.csv'""")

## observed problems
- customers.csv
    - signup_date column contains bad data (month 13)
    - country column is mix of upper and lower case strings, some are empty
- subscriptions.csv
    - end_date column contains bad data (february 30th)
    - monthly price is mostly numbers, but contains also the string "thirty"
    - mispellings present in plan column
    - reference to non-existing customer C999

In [ ]:
conn.execute("SUMMARIZE customers").df()

In [ ]:
conn.execute("""DROP TABLE IF EXISTS customers_clean;""")
conn.execute("""
CREATE TABLE IF NOT EXISTS customers_clean AS (
SELECT
    customer_id::VARCHAR AS customer_id,
    try_cast(signup_date AS DATE) AS signup_date,  -- this takes out our bad dates
    upper(country)::VARCHAR
FROM customers);
""")
# conn.execute("SELECT * FROM customers_clean where customer_id in ['C019', 'C027', 'C033'];").df()

In [ ]:
conn.execute("SUMMARIZE subscriptions").df()

In [ ]:
conn.execute("""DROP TABLE IF EXISTS subscriptions_clean;""")
conn.execute("""
CREATE TABLE IF NOT EXISTS subscriptions_clean AS (
SELECT
    customer_id::VARCHAR as customer_id,
    try_cast(start_date AS DATE) as start_date,
    try_cast(end_date AS DATE) as end_date,
    try_cast(monthly_price AS INTEGER) as monthly_price
FROM subscriptions
);
""")
conn.execute("""SUMMARIZE subscriptions_clean;""").df()

## Assumptions
- signup_date in customers table must be before start_date of subscription
- end_date must be after start_date of subscription
- data problems should be flagged rather than silently repaired, calculations should disregard bad data
- ...